# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:

initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
rows_affected = initial_rows - df.shape[0]
log(1, 'Dropped duplicate rows', rows_affected)
display(df.head())

[1] Dropped duplicate rows (15 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 2 — clean `price` -> float

In [4]:
initial_price_dtype = df['price'].dtype
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False)
df['price'] = pd.to_numeric(df['price'])
final_price_dtype = df['price'].dtype

if initial_price_dtype != final_price_dtype:
    log(2, 'Cleaned price column to float', 0) # 0 rows affected
else:
    log(2, 'Price column already clean (no change in dtype)', 0)
display(df.head())

[2] Cleaned price column to float (0 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,7.5
3,3,Cheeseburger,Food,NaN,7.5
4,4,cheese burger,Apparel,1.0,7.5


\### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty:

In [5]:
initial_rows = df.shape[0]

df['qty'] = pd.to_numeric(df['qty'], errors='coerce')

rows_to_drop_nan = df['qty'].isna().sum()
rows_to_drop_negative = df[df['qty'] < 0].shape[0]

df.dropna(subset=['qty'], inplace=True)
df = df[df['qty'] >= 0]

rows_affected = initial_rows - df.shape[0]
log(3, 'Cleaned qty column: converted to numeric, dropped missing/negative qty', rows_affected)
display(df.head())

[3] Cleaned qty column: converted to numeric, dropped missing/negative qty (25 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,7.5
4,4,cheese burger,Apparel,1.0,7.5
5,5,Foam Finger,food,3.0,6.0


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
print(df['item'].value_counts())

#define mapping
ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'rain poncho': 'Rain Poncho'
}

initial_unique_items = df['item'].nunique()
#  mapping
df['item'] = df['item'].replace(ITEM_MAP)
final_unique_items = df['item'].nunique()

log(4, f'Canonicalized item column: collapsed {initial_unique_items - final_unique_items} unique item variants', rows_affected)
display(df.head())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[4] Canonicalized item column: collapsed 3 unique item variants (25 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,Foam Finger,Apparel,1.0,7.5
2,2,Cheeseburger,Merch,1.0,7.5
4,4,Cheeseburger,Apparel,1.0,7.5
5,5,Foam Finger,food,3.0,6.0


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
print(df['category'].value_counts())

#  mapping for category column
CATEGORY_MAP = {
    'food': 'Food',
    'rain-gear': 'RainGear',
    'Apparel': 'Merch' # Business decision to merge Apparel into Merch
}

initial_unique_categories = df['category'].nunique()

#  mapping
df['category'] = df['category'].replace(CATEGORY_MAP)

final_unique_categories = df['category'].nunique()

rows_affected = 0
log(5, f'normalized category column: collapsed {initial_unique_categories - final_unique_categories} unique category variants, including merging Apparel into Merch', rows_affected)
display(df.head())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[5] normalized category column: collapsed 3 unique category variants, including merging Apparel into Merch (0 row(s))


,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,12.0
1,1,Foam Finger,Merch,1.0,7.5
2,2,Cheeseburger,Merch,1.0,7.5
4,4,Cheeseburger,Merch,1.0,7.5
5,5,Foam Finger,Food,3.0,6.0


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].nunique() == final_unique_items
assert df['category'].nunique() == final_unique_categories
print('clean:', df.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
df['revenue'] = df['qty'] * df['price']

# revenue by category
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).round(2)
print('Revenue by Category:')
print(revenue_by_category)

# overall total revenue
total_revenue = df['revenue'].sum().round(2)
print(f'\noverall Total Revenue: ${total_revenue:.2f}')

print('\nfocus on stocking more ' + revenue_by_category.index[0] + ' items, as they generate the highest revenue.')

Revenue by Category:
category
Food        1656.0
Merch       1572.0
RainGear    1512.0
Name: revenue, dtype: float64

overall Total Revenue: $4740.00

focus on stocking more Food items, as they generate the highest revenue.


**What I would tell the vendor:** _..._

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1,Dropped duplicate rows,15
1,2,Cleaned price column to float,0
2,3,"Cleaned qty column: converted to numeric, drop...",25
3,4,Canonicalized item column: collapsed 3 unique ...,25
4,5,normalized category column: collapsed 3 unique...,0


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

a) The cleaning step that changed the revenue total the most was step 3, which involved cleaning the qty column and dropping rows with missing/negative quantities, which removed 25 rows from the df.

revenue before step 3, 5171.00 estimated

revenue after step 3, 4740.00

b) One decision where a reasonable person could have chosen differently was in todo 5, where Apparel was merged into Merch as a business decision. An alternative choice would have been to keep Apparel as a distinct category, because the overall total revenue ($4740.00) would have remained unchanged, because this altered category labels and did not affect quantities/ prices/the number of rows.

I chose to merge Apparel into Merch based on the assumption that Apparel items are a subset of Merch, so this would reduce the number of distinct categories and make the data more simple_your answer here_